# 실습 4 · 선형 회귀 직접 만들기

**인공지능 일반**

코드의 `여기를_채우세요` ①~⑥을 채운 뒤 위에서부터 순서대로 실행하세요.

## 0. 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 그래프에 한글을 쓰기 위한 준비 (실패해도 실습은 진행됩니다) ---
try:
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum > /dev/null 2>&1
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 설정 실패 — 그래프의 한글이 □□로 보일 수 있습니다.')

plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터

`x` : TV 광고비 (백만 원), `y` : 매출 (억 원). 실제 관계는 `y = 7 + 4.8·x + ε` 입니다. 훈련 80개, 테스트 20개로 나눕니다.

In [ ]:
rng = np.random.default_rng(0)
N = 100
x = rng.uniform(0, 3, N)
y = 7 + 4.8 * x + rng.normal(0, 1.2, N)

idx = rng.permutation(N)
train, test = idx[:80], idx[80:]
x_train, y_train = x[train], y[train]
x_test,  y_test  = x[test],  y[test]

plt.scatter(x_train, y_train, s=18, label='훈련')
plt.scatter(x_test,  y_test,  s=18, marker='s', label='테스트')
plt.xlabel('TV 광고비 (백만 원)'); plt.ylabel('매출 (억 원)')
plt.legend(); plt.show()

## 2. STEP 1 · 모델 — 직선 ŷ = a·x + b

In [ ]:
def predict(x, a, b):
    # ✏️ 실습 ①
    return 여기를_채우세요

## 3. STEP 2 · 채점 기준 — RSS, MSE

$$\text{RSS} = \sum_{i=1}^{n}(y_i - \hat y_i)^2, \qquad \text{MSE} = \frac{\text{RSS}}{n}$$

In [ ]:
def rss(y, y_hat):
    # ✏️ 실습 ② : (정답 − 예측)² 의 합
    return 여기를_채우세요


def mse(y, y_hat):
    return 여기를_채우세요     # RSS를 데이터 개수로 나눈 평균


# 손 계산 확인: RSS 6, MSE 1.5
hx = np.array([1, 2, 3, 4])
hy = np.array([3, 5, 4, 8])
print('RSS =', rss(hy, predict(hx, 2, 0)), '  MSE =', mse(hy, predict(hx, 2, 0)))

## 4. STEP 3-① · 공식으로 한 번에

$$a = \frac{\sum (x_i-\bar x)(y_i-\bar y)}{\sum (x_i-\bar x)^2}, \qquad b = \bar y - a\,\bar x$$

In [ ]:
def fit_formula(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    # ✏️ 실습 ③
    a = 여기를_채우세요
    b = 여기를_채우세요
    return a, b


a_f, b_f = fit_formula(x_train, y_train)
print(f'ŷ = {a_f:.3f}·x + {b_f:.3f}')

xs = np.linspace(0, 3, 50)
plt.scatter(x_train, y_train, s=18)
plt.plot(xs, predict(xs, a_f, b_f), color='red')
plt.xlabel('TV 광고비 (백만 원)'); plt.ylabel('매출 (억 원)'); plt.show()

### 정규방정식

$$\boldsymbol\beta = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{Y}, \qquad \mathbf{X} = [\,1,\ x\,],\ \boldsymbol\beta = [\,b,\ a\,]$$

In [ ]:
def fit_normal_eq(x, y):
    X = np.column_stack([np.ones(len(x)), x])
    # ✏️ 실습 ④ : 힌트: np.linalg.inv(…),  행렬 곱은 @,  전치는 X.T
    return 여기를_채우세요


beta = fit_normal_eq(x_train, y_train)
print(f'정규방정식:  b = {beta[0]:.3f},  a = {beta[1]:.3f}')
print(f'공식      :  b = {b_f:.3f},  a = {a_f:.3f}')

## 5. STEP 3-② · 경사하강법

$$a \leftarrow a - \eta\,\frac{\partial\,\text{MSE}}{\partial a}, \qquad b \leftarrow b - \eta\,\frac{\partial\,\text{MSE}}{\partial b}$$

$$\frac{\partial\,\text{MSE}}{\partial a} = -\frac{2}{n}\sum x_i (y_i - \hat y_i), \qquad \frac{\partial\,\text{MSE}}{\partial b} = -\frac{2}{n}\sum (y_i - \hat y_i)$$

In [ ]:
def fit_gd(x, y, lr=0.05, steps=2000):
    a, b = 0.0, 0.0
    history = []
    for _ in range(steps):
        error = y - predict(x, a, b)
        # ✏️ 실습 ⑤
        grad_a = 여기를_채우세요
        grad_b = 여기를_채우세요
        a = 여기를_채우세요
        b = 여기를_채우세요
        history.append(mse(y, predict(x, a, b)))
    return a, b, history


a_g, b_g, hist = fit_gd(x_train, y_train)
print(f'경사하강법:  ŷ = {a_g:.3f}·x + {b_g:.3f}')
print(f'공식      :  ŷ = {a_f:.3f}·x + {b_f:.3f}')

plt.plot(hist); plt.yscale('log')
plt.xlabel('걸음 수'); plt.ylabel('훈련 MSE'); plt.show()

## 6. STEP 4 · 평가 — RMSE, R²

$$\text{RMSE} = \sqrt{\text{MSE}}, \qquad R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}, \quad \text{TSS} = \sum (y_i - \bar y)^2$$

In [ ]:
def r2(y, y_hat):
    # ✏️ 실습 ⑥
    rss = 여기를_채우세요
    tss = 여기를_채우세요
    return 여기를_채우세요


y_hat_test = predict(x_test, a_f, b_f)
print(f'테스트 RMSE = {np.sqrt(mse(y_test, y_hat_test)):.3f} 억 원')
print(f'테스트 R²   = {r2(y_test, y_hat_test):.3f}')